In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,mean_squared_error
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack


In [4]:
train_df=pd.read_csv("/content/train.xlsx - Dataset_120.csv")
test_df=pd.read_csv("/content/test.xlsx - Sheet1.csv")


In [5]:
train_df.columns

Index(['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours',
       'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood',
       'face_emotion_hint', 'reflection_quality', 'emotional_state',
       'intensity'],
      dtype='object')

In [6]:
train_df["journal_text"] = train_df["journal_text"].fillna("")
test_df["journal_text"] = test_df["journal_text"].fillna("")

train_df["energy_level"] = train_df["energy_level"].fillna(3)
test_df["energy_level"] = test_df["energy_level"].fillna(3)

train_df["stress_level"] = train_df["stress_level"].fillna(3)
test_df["stress_level"] = test_df["stress_level"].fillna(3)

train_df["sleep_hours"] = train_df["sleep_hours"].fillna(train_df["sleep_hours"].median())
test_df["sleep_hours"] = test_df["sleep_hours"].fillna(test_df["sleep_hours"].median())


In [7]:
time_map = {"morning":0, "afternoon":1, "evening":2, "night":3}
mood_map = {"happy":2, "calm":1, "neutral":0, "sad":-1, "stressed":-2}

train_df["time_of_day"] = train_df["time_of_day"].map(time_map)
test_df["time_of_day"] = test_df["time_of_day"].map(time_map)

train_df["previous_day_mood"] = train_df["previous_day_mood"].map(mood_map)
test_df["previous_day_mood"] = test_df["previous_day_mood"].map(mood_map)


In [8]:
le_amb = LabelEncoder()
train_df["ambience_type"] = le_amb.fit_transform(train_df["ambience_type"].astype(str))
test_df["ambience_type"] = le_amb.transform(test_df["ambience_type"].astype(str))

le_face = LabelEncoder()
train_df["face_emotion_hint"] = le_face.fit_transform(train_df["face_emotion_hint"].astype(str))
test_df["face_emotion_hint"] = le_face.transform(test_df["face_emotion_hint"].astype(str))

In [9]:
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X_text = tfidf.fit_transform(train_df["journal_text"])


In [10]:
meta_cols = [
    "duration_min",
    "sleep_hours",
    "energy_level",
    "stress_level",
    "time_of_day",
    "previous_day_mood",
    "ambience_type",
    "face_emotion_hint"
]

X_meta = train_df[meta_cols].fillna(0).values

X_full = hstack([X_text, X_meta])

In [11]:
le_state = LabelEncoder()
y_state = le_state.fit_transform(train_df["emotional_state"])
y_intensity = train_df["intensity"]


In [12]:
X_tr, X_val, y_tr_state, y_val_state = train_test_split(
    X_full, y_state, test_size=0.2, random_state=42
)

_, _, y_tr_int, y_val_int = train_test_split(
    X_full, y_intensity, test_size=0.2, random_state=42
)

In [13]:
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
lr_clf = LogisticRegression(max_iter=1000)

rf_clf.fit(X_tr, y_tr_state)
lr_clf.fit(X_tr, y_tr_state)

rf_pred = rf_clf.predict(X_val)
lr_pred = lr_clf.predict(X_val)

print("\n RANDOM FOREST PERFORMANCE")
print(classification_report(y_val_state, rf_pred))
print("\n LOGISTIC REGRESSION PERFORMANCE")
print(classification_report(y_val_state, lr_pred))
state_model = rf_clf


 RANDOM FOREST PERFORMANCE
              precision    recall  f1-score   support

           0       0.75      0.71      0.73        51
           1       0.67      0.71      0.69        41
           2       0.73      0.61      0.67        36
           3       0.82      0.61      0.70        46
           4       0.75      0.73      0.74        37
           5       0.45      0.76      0.56        29

    accuracy                           0.68       240
   macro avg       0.70      0.69      0.68       240
weighted avg       0.71      0.68      0.69       240


 LOGISTIC REGRESSION PERFORMANCE
              precision    recall  f1-score   support

           0       0.75      0.53      0.62        51
           1       0.67      0.68      0.67        41
           2       0.67      0.61      0.64        36
           3       0.84      0.59      0.69        46
           4       0.63      0.65      0.64        37
           5       0.36      0.72      0.48        29

    accuracy   

In [17]:
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
    X_full, y_intensity, test_size=0.2, random_state=42
)

intensity_model = RandomForestRegressor(n_estimators=200, random_state=42)
intensity_model.fit(X_tr2, y_tr2)

int_pred = intensity_model.predict(X_val2)

print("\n INTENSITY MSE:", mean_squared_error(y_val2, int_pred))


 INTENSITY MSE: 2.1940746875


In [18]:
state_model.fit(X_full, y_state)
intensity_model.fit(X_full, y_intensity)


RandomForestRegressor(n_estimators=200, random_state=42)

In [15]:

try:
    from xgboost import XGBRegressor

    intensity_model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    print("\n Using XGBoost for intensity")

except:
    from sklearn.ensemble import GradientBoostingRegressor

    intensity_model = GradientBoostingRegressor()

    print("\n XGBoost not available → using GradientBoosting")

# Train-validation split
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
    X_full, y_intensity, test_size=0.2, random_state=42
)

# Train
intensity_model.fit(X_tr2, y_tr2)

# Predict
int_pred = intensity_model.predict(X_val2)

# Clip predictions (IMPORTANT)
int_pred = np.clip(int_pred, 1, 5)

# Evaluate
mse = mean_squared_error(y_val2, int_pred)
rmse = np.sqrt(mse)

print("\n INTENSITY PERFORMANCE")
print("MSE:", mse)
print("RMSE:", rmse)


 Using XGBoost for intensity

 INTENSITY PERFORMANCE
MSE: 2.393052577972412
RMSE: 1.5469494426038661


In [19]:
X_text_test = tfidf.transform(test_df["journal_text"])
X_meta_test = test_df[meta_cols].fillna(0).values
X_test = hstack([X_text_test, X_meta_test])


In [21]:
state_probs = state_model.predict_proba(X_test)
state_preds_idx = np.argmax(state_probs, axis=1)
state_preds = le_state.inverse_transform(state_preds_idx)
intensity_preds = intensity_model.predict(X_test)



In [22]:
confidence = np.max(state_probs, axis=1)
uncertain_flag = (confidence < 0.6).astype(int)



In [23]:
def decide_action(state, stress, energy, time_of_day, prev_mood):

    if stress >= 4 and energy <= 2:
        return "box_breathing", "now"

    if state == "sad" and prev_mood < 0:
        return "journaling", "tonight"

    if energy >= 4 and time_of_day in [0,1]:
        return "deep_work", "within_15_min"

    if time_of_day == 3:
        return "rest", "now"

    if stress >= 3:
        return "grounding", "within_15_min"

    return "light_planning", "later_today"



In [24]:
def generate_message(state):

    if state == "stressed":
        return "You seem stressed. Try a short breathing exercise."

    if state == "sad":
        return "You seem low. Writing your thoughts may help."

    if state == "happy":
        return "You're in a good state. Use this time productively."

    return "Take a moment to stay balanced."



In [25]:
actions, times, messages = [], [], []

for i in range(len(test_df)):

    action, timing = decide_action(
        state_preds[i],
        test_df["stress_level"].iloc[i],
        test_df["energy_level"].iloc[i],
        test_df["time_of_day"].iloc[i],
        test_df["previous_day_mood"].iloc[i]
    )

    msg = generate_message(state_preds[i])

    actions.append(action)
    times.append(timing)
    messages.append(msg)

In [27]:
output = pd.DataFrame({
    "id": test_df["id"],
    "predicted_state": state_preds,
    "predicted_intensity": intensity_preds,
    "confidence": confidence,
    "uncertain_flag": uncertain_flag,
    "what_to_do": actions,
    "when_to_do": times,
    "support_message": messages
})

output.to_csv("predictions.csv", index=False)

print("\n FINAL predictions.csv generated!")


 FINAL predictions.csv generated!


This example demonstrates how the system takes user input and provides emotional understanding and guidance.

In [43]:
# ==========================================
# 🌿 SIMPLE DEMO (ERROR-FREE)
# ==========================================

print("\n✅ Running Demo Example...\n")

# -----------------------------
# Example Input
# -----------------------------
journal = "I feel very tired and stressed today, nothing seems to go right"

sleep = 5
energy = 2
stress = 5
time_of_day = 0        # morning
previous_mood = -1     # sad

# NOTE:
# We are directly using numbers for encoded values
# (to avoid ANY encoding error)

ambience_val = 0
face_val = 0

print("🧾 Journal:", journal)

# -----------------------------
# Transform Input
# -----------------------------
text_vec = tfidf.transform([journal])

meta = np.array([[
    10,              # duration_min
    sleep,
    energy,
    stress,
    time_of_day,
    previous_mood,
    ambience_val,
    face_val
]])

X_input = hstack([text_vec, meta])

# -----------------------------
# Prediction
# -----------------------------
prob = state_model.predict_proba(X_input)
state = le_state.inverse_transform([np.argmax(prob)])[0]

intensity = intensity_model.predict(X_input)[0]
intensity = np.clip(intensity, 1, 5)

confidence = np.max(prob)
uncertain = int(confidence < 0.6)

# -----------------------------
# Decision Engine
# -----------------------------
def decide_action(state, stress, energy, time_of_day, prev_mood):
    if stress >= 4 and energy <= 2:
        return "box_breathing", "now"
    if state == "sad" and prev_mood < 0:
        return "journaling", "tonight"
    if energy >= 4 and time_of_day in [0,1]:
        return "deep_work", "within_15_min"
    if time_of_day == 3:
        return "rest", "now"
    if stress >= 3:
        return "grounding", "within_15_min"
    return "light_planning", "later_today"

def generate_message(state):
    if state == "stressed":
        return "You seem stressed. Try a short breathing exercise."
    if state == "sad":
        return "You seem low. Writing your thoughts may help."
    if state == "happy":
        return "You're in a good state. Use this time productively."
    return "Take a moment to stay balanced."

action, timing = decide_action(state, stress, energy, time_of_day, previous_mood)
message = generate_message(state)

# -----------------------------
# Output
# -----------------------------
print("\n🧠 MODEL OUTPUT")
print("Predicted State:", state)
print("Intensity:", round(float(intensity),2))
print("Confidence:", round(float(confidence),2))
print("Uncertain Flag:", uncertain)

print("\n🎯 RECOMMENDATION")
print("What to do:", action)
print("When:", timing)

print("\n💬 MESSAGE")
print(message)


✅ Running Demo Example...

🧾 Journal: I feel very tired and stressed today, nothing seems to go right

🧠 MODEL OUTPUT
Predicted State: neutral
Intensity: 3.48
Confidence: 0.54
Uncertain Flag: 1

🎯 RECOMMENDATION
What to do: box_breathing
When: now

💬 MESSAGE
Take a moment to stay balanced.
